# 모델 학습 핵심 정리fit() 파라미터, 콜백함수(EarlyStopping/ModelCheckpoint/TensorBoard)

## 1. `model.fit()` 핵심 파라미터 — 학습을 어떻게 돌릴지 결정하는 옵션들

In [ ]:
import tensorflow as tffrom tensorflow import kerasmodel.fit(  x_train, y_train,          # 입력 데이터, 정답 라벨  batch_size=32,              # 한 번에 넣을 데이터 개수  epochs=10,                  # 전체 데이터를 몇 바퀴 돌릴지  verbose=1,                  # 0: 출력 없음, 1: 진행바, 2: epoch당 한 줄  validation_data=(x_val, y_val),  # 매 epoch마다 검증할 데이터  validation_freq=1,          # 몇 epoch마다 검증할지  shuffle=True,                # epoch마다 데이터 섞을지  class_weight=None,           # 클래스 불균형 있을 때 클래스별 가중치 줄 수 있음  initial_epoch=0,             # 이어서 학습할 때 시작 epoch 번호 (resume용)  workers=1,                   # 데이터 불러올 때 쓸 프로세스 수  use_multiprocessing=False    # 멀티프로세싱으로 데이터 로드할지)

## 2. 콜백함수 — 학습 도중(epoch 시작/끝 등) 특정 시점에 자동으로 실행되는 함수

In [ ]:
class MyCallback(keras.callbacks.Callback):  # epoch 하나가 끝날 때마다 자동으로 호출됨  def on_epoch_end(self, epoch, logs=None):    print(f"{epoch} epoch 끝, loss={logs['loss']}")  # logs에 그 epoch의 결과가 담겨있음model.fit(x_train, y_train, epochs=10, callbacks=[MyCallback()])  # callbacks 리스트에 넣어서 전달

### 2-1. EarlyStopping — 성능 개선이 없으면 학습을 자동으로 멈춤 (과적합 방지)

In [ ]:
early_stop = keras.callbacks.EarlyStopping(  monitor='val_loss',   # 어떤 값을 기준으로 볼지 (검증 손실)  min_delta=0.001,       # 이 정도는 변해야 '개선'으로 인정  patience=3,            # 개선이 없어도 몇 epoch까지는 기다릴지  mode='min',             # val_loss는 작을수록 좋으니 'min' (정확도라면 'max')  restore_best_weights=True  # 멈출 때 제일 좋았던 시점의 가중치로 복원)model.fit(x_train, y_train, epochs=100, validation_data=(x_val, y_val), callbacks=[early_stop])

### 2-2. ModelCheckpoint — 학습 중간중간 모델(가중치)을 파일로 자동 저장

In [ ]:
checkpoint = keras.callbacks.ModelCheckpoint(  filepath='checkpoints/cp-{epoch:04d}.ckpt',  # {epoch:04d} 자리에 epoch 번호가 자동으로 채워짐  save_best_only=True,   # 성능 제일 좋을 때만 저장 (아니면 매번 덮어씀/누적 저장)  save_weights_only=True  # True면 가중치만 저장(가벼움), False면 모델 구조까지 통째로 저장)model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val), callbacks=[checkpoint])

### 2-3. TensorBoard — 학습 로그를 시각적으로 확인하는 도구용 콜백

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(  log_dir="./logs",   # 로그를 쌓을 폴더  update_freq='epoch'  # 얼마마다 로그 기록할지)model.fit(x_train, y_train, epochs=10, callbacks=[tensorboard_callback])# 터미널에서 아래 명령으로 로그 확인 (주피터 셀 아님, CLI)# tensorboard --logdir=./logs